# Silver to Gold v1 — posted sales

Five outputs: `fact_posted_sales`, `dim_customer`, `dim_item`, `dim_salesperson`, and `dim_date`.

## First run
1. Create the schema-enabled **LH_SALES_GOLD_TEST** lakehouse in the same workspace.
2. Import this notebook. Attach **LH_SALES_GOLD_TEST** as its default lakehouse and
   attach **LH_SALES_SILVER_TEST** as an additional lakehouse. Restart an existing
   Spark session after changing its default lakehouse.
3. Finish Silver loads first. Do not overlap Silver/Gold writers. Keep `WRITE_GOLD=False`
   and run all cells. This mode reads Silver, checks the proposed Gold outputs, and
   prints aggregate totals; it writes no Gold tables or audit rows.
4. Review counts, currencies, placeholders, dates and the credit-memo sign convention.
   After validation passes, set `WRITE_GOLD=True` and run all to create/replace the five
   tables under **LH_SALES_GOLD_TEST.gold** and append Gold run audit events.

The first build uses eight existing Silver tables. SharePoint, historical actuals,
budget, forecast, Security ACL, open orders, custom item/customer extensions and item
references are deferred. This is not a reproduction of the complete existing model.

## Business rules to review
- One row per company, posted document type, document number and line number.
- Posting date comes from the header. Type 0 comment lines with the observed default
  line date `1753-01-01` use that valid header date and are flagged. The original line
  date is preserved. Other date mismatches, including null line dates, fail validation.
- Header bill-to customer is the default reporting customer. Sell-to customer is also
  available. Blank/null customer codes on type 0 comment lines may inherit a populated
  header customer, regardless of line date. Original line customer codes and separate
  bill-to/sell-to default flags are retained. Populated mismatches on comment lines and
  all customer mismatches on financial lines still fail. Header salesperson represents document attribution; customer dimension
  salesperson is the current master assignment, not its historical value.
- Invoice signs are preserved; credit values are **negated, never ABS-converted**.
  Negative credit lines therefore become positive. Source amounts are retained.
- Amounts excluding VAT and including VAT remain separate. Currency is preserved;
  blank currency code means company local currency. **No currency conversion** or
  cross-currency revenue total is produced. Quantities also require appropriate UOM grouping.
- All posted line types are retained, including comments, G/L and charges. The
  `is_financial_line` flag identifies `type != 0`; downstream revenue measures must
  agree on scope. Source cancellation, correction and prepayment flags are retained,
  not used as automatic exclusions. The notebook does not infer finance policy.
- Standard BC item line type `2` links to the item dimension. Non-item lines have an
  explicit not-applicable member. Missing masters get flagged placeholder members;
  missing document headers fail instead of dropping lines.
- Keys use `K:` plus the source code; `U:` means unknown and `N:` not applicable.
  Current dimensions are fully replaced (Type 1), with no historical attributes.
- Date dimension covers full calendar years of posting dates and the current UTC year.
  Fiscal calendar logic and alternate document-date relationships are deferred.

## Operational contract
Gold v1 requires successful audited **full Silver loads** from v6. It pins each
source Delta version and records those versions, batch IDs and counts in the Gold
audit. Per-table versions do not constitute a coordinated source-system snapshot.
The current Silver values must match their latest successful audit entries.

All transformations, keys, joins and source amount reconciliations are checked before
Gold table writes. Existing Gold schemas must match exactly by name and type.
Writes are independent Delta transactions: a failure can leave partially updated Gold.
Only update an Import semantic model after the notebook succeeds; Direct Lake readers
could see partial updates during writes and require a separate publication strategy.
`_gold_load_audit` records RUNNING/SUCCESS/FAILED events and the tables already written.
No scheduler, semantic model or production report is changed by this notebook.

Local synthetic Spark tests cover transformations. Fabric lakehouse access, Delta
writes, audit persistence and the actual company data require the first Fabric run.


## 1. Configuration — keep validation-only mode for the first run

In [ ]:
from datetime import date, datetime, timezone
from decimal import Decimal
from uuid import uuid4
import json
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark import StorageLevel

SOURCE_LAKEHOUSE = "LH_SALES_SILVER_TEST"
SOURCE_SCHEMA = "silver_clean"
TARGET_LAKEHOUSE = "LH_SALES_GOLD_TEST"
TARGET_SCHEMA = "gold"
WRITE_GOLD = False  # First run validates only. Set True after reviewing the results.
COMPANY_KEY = "ACCREDO_PACKAGING_INC"  # Stable identifier for this one source company.
ITEM_LINE_TYPE = 2  # Business Central standard Item line type; verify for this source.
MIN_POSTING_YEAR = 1900
MAX_POSTING_YEAR = 2100

SOURCE_TABLES = [
    "sales_invoice_header", "sales_invoice_line",
    "sales_credit_memo_header", "sales_credit_memo_line",
    "customer", "item", "item_category", "salesperson_purchaser",
]
OUTPUT_KEYS = {
    "dim_customer": ["company_key", "customer_key"],
    "dim_item": ["company_key", "item_key"],
    "dim_salesperson": ["company_key", "salesperson_key"],
    "dim_date": ["date_key"],
    "fact_posted_sales": ["company_key", "document_type", "document_no", "line_no"],
}


## 2. Transformations, validation, source versioning and publication

In [ ]:
def configure_spark():
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    spark.conf.set("spark.sql.ansi.enabled", "true")
    # Consistent with the validated Silver notebook's calendar convention.
    for suffix in ["datetimeRebaseModeInRead", "int96RebaseModeInRead",
                   "datetimeRebaseModeInWrite", "int96RebaseModeInWrite"]:
        spark.conf.set("spark.sql.parquet." + suffix, "CORRECTED")
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")


def require_columns(df, names, label):
    missing = sorted(set(names) - set(df.columns))
    if missing:
        raise ValueError(f"{label}: missing columns {missing}")


def reject_rows(df, predicate, label):
    if df.filter(predicate).limit(1).count():
        raise ValueError(label)


def validate_key(df, keys, label):
    require_columns(df, keys, label)
    invalid = F.lit(False)
    for k in keys:
        invalid = invalid | F.col(k).isNull() | (F.trim(F.col(k).cast("string")) == "")
    reject_rows(df, invalid, f"{label}: blank/null key")
    if df.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{label}: duplicate key {keys}")


def text_code(column):
    # Strip surrounding ordinary spaces; do not change case or leading zeroes.
    return F.trim(column.cast("string"))


def relation_key(column):
    # Prefix real keys so source values cannot collide with the unknown member.
    c = text_code(column)
    return F.when(c.isNull() | (c == ""), F.lit("U:")).otherwise(F.concat(F.lit("K:"), c))


def require_money(df, names, label):
    require_columns(df, names, label)
    for name in names:
        if not isinstance(df.schema[name].dataType, DecimalType):
            raise ValueError(f"{label}.{name}: expected an exact decimal type")
        reject_rows(df, F.col(name).isNull(), f"{label}.{name}: null numeric value")


def build_document(lines, headers, document_type):
    if document_type not in {"INVOICE", "CREDIT_MEMO"}:
        raise ValueError("Unsupported document type")
    require_columns(lines, ["document_no", "line_no", "type", "no", "description",
                           "quantity", "quantity_base", "unit_of_measure_code", "amount",
                           "amount_including_vat", "posting_date", "bill_to_customer_no",
                           "sell_to_customer_no", "_silver_run_id"], document_type + " lines")
    require_columns(headers, ["no", "posting_date", "document_date", "bill_to_customer_no",
                             "sell_to_customer_no", "salesperson_code", "currency_code",
                             "currency_factor", "correction", "marked_as_canceled",
                             "prepayment_invoice" if document_type == "INVOICE" else "prepayment_credit_memo",
                             "_silver_run_id"], document_type + " headers")
    require_money(lines, ["quantity", "quantity_base", "amount", "amount_including_vat"], document_type)
    l = lines.withColumn("document_no", text_code(F.col("document_no")))
    h = headers.withColumn("no", text_code(F.col("no")))
    validate_key(l, ["document_no", "line_no"], document_type + " lines")
    validate_key(h, ["no"], document_type + " headers")
    reject_rows(l, F.col("type").isNull(), document_type + ": missing line type")
    if l.join(h.select(F.col("no").alias("document_no")), "document_no", "left_anti").limit(1).count():
        raise ValueError(document_type + ": line without matching header")
    j = l.alias("l").join(h.alias("h"), F.col("l.document_no") == F.col("h.no"), "inner")
    reject_rows(j, F.to_date(F.col("h.posting_date")).isNull(), document_type + ": missing posting date")
    reject_rows(j, ~F.year(F.col("h.posting_date")).between(MIN_POSTING_YEAR, MAX_POSTING_YEAR),
                document_type + ": posting year outside configured range")
    # Observed BC comment lines may carry the SQL default date 1753-01-01.
    # Accept only that specific sentinel on type 0; financial lines remain strict.
    line_date = F.to_date(F.col("l.posting_date"))
    comment_default_date = F.coalesce(
        (F.col("l.type") == 0) & (line_date == F.lit(date(1753, 1, 1))),
        F.lit(False),
    )
    date_mismatch = line_date.isNull() | (line_date != F.to_date(F.col("h.posting_date")))
    reject_rows(j, date_mismatch & ~comment_default_date,
                document_type + ": line/header posting dates differ")
    # Comment lines can omit customer codes regardless of their posting date.
    # Only blank/null comment-line codes may inherit a populated header code.
    customer_defaulted = {}
    for c in ["bill_to_customer_no", "sell_to_customer_no"]:
        line_key = relation_key(F.col("l." + c))
        header_key = relation_key(F.col("h." + c))
        customer_defaulted[c] = (
            (F.col("l.type") == 0) & (line_key == "U:") & (header_key != "U:")
        )
        reject_rows(j, (~line_key.eqNullSafe(header_key)) & ~customer_defaulted[c],
                    document_type + f": line/header {c} differs")
    credit = document_type == "CREDIT_MEMO"
    def signed(name):
        # Unary negation preserves decimal precision and correctly reverses negative credits.
        return -F.col("l." + name) if credit else F.col("l." + name)
    prepayment = "prepayment_credit_memo" if credit else "prepayment_invoice"
    return j.select(
        F.lit(COMPANY_KEY).alias("company_key"), F.lit(document_type).alias("document_type"),
        F.col("l.document_no"), F.col("l.line_no"), F.col("l.type").alias("line_type"),
        (F.col("l.type") != 0).alias("is_financial_line"),
        F.col("l.no").alias("source_line_no"), F.col("l.description"),
        line_date.alias("source_line_posting_date"),
        comment_default_date.alias("line_posting_date_defaulted"),
        F.to_date(F.col("h.posting_date")).alias("posting_date"),
        F.date_format(F.col("h.posting_date"), "yyyyMMdd").cast("int").alias("posting_date_key"),
        F.to_date(F.col("h.document_date")).alias("document_date"),
        F.col("l.bill_to_customer_no").alias("source_line_bill_to_customer_no"),
        F.col("l.sell_to_customer_no").alias("source_line_sell_to_customer_no"),
        customer_defaulted["bill_to_customer_no"].alias("bill_to_customer_defaulted"),
        customer_defaulted["sell_to_customer_no"].alias("sell_to_customer_defaulted"),
        relation_key(F.col("h.bill_to_customer_no")).alias("bill_to_customer_key"),
        relation_key(F.col("h.sell_to_customer_no")).alias("sell_to_customer_key"),
        relation_key(F.col("h.salesperson_code")).alias("salesperson_key"),
        F.when(F.col("l.type") == ITEM_LINE_TYPE, relation_key(F.col("l.no")))
            .otherwise(F.lit("N:")).alias("item_key"),
        F.coalesce(text_code(F.col("h.currency_code")), F.lit("")).alias("currency_code"),
        F.col("h.currency_factor"), F.col("h.correction").alias("source_correction"),
        F.col("h.marked_as_canceled").alias("source_marked_as_canceled"),
        F.col("h." + prepayment).alias("source_prepayment"),
        F.col("l.unit_of_measure_code"),
        F.col("l.quantity").alias("source_quantity"), signed("quantity").alias("signed_quantity"),
        F.col("l.quantity_base").alias("source_quantity_base"), signed("quantity_base").alias("signed_quantity_base"),
        F.col("l.amount").alias("source_amount_excl_vat"), signed("amount").alias("signed_amount_excl_vat"),
        F.col("l.amount_including_vat").alias("source_amount_incl_vat"),
        signed("amount_including_vat").alias("signed_amount_incl_vat"),
        F.col("l._silver_run_id").alias("_silver_line_run_id"),
        F.col("h._silver_run_id").alias("_silver_header_run_id"),
    )


def add_missing_members(dim, key, name_column, references):
    # Keep every fact key, including deleted master records, with an explicit placeholder.
    wanted = references.select(F.col(key)).distinct()
    missing = wanted.join(dim.select(key), key, "left_anti")
    placeholders = missing.select(*[
        F.col(key) if c == key else
        F.lit(COMPANY_KEY).alias(c) if c == "company_key" else
        F.when(F.col(key) == "N:", F.lit("Not applicable"))
         .when(F.col(key) == "U:", F.lit("Unknown / unassigned"))
         .otherwise(F.concat(F.lit("Missing master: "), F.col(key))).alias(c) if c == name_column else
        F.lit(True).alias(c) if c == "is_placeholder" else
        F.lit(None).cast(dim.schema[c].dataType).alias(c)
        for c in dim.columns
    ])
    return dim.unionByName(placeholders)


def build_dimensions(sources, fact):
    customer = sources["customer"]
    require_columns(customer, ["no", "name", "city", "county", "post_code", "country_region_code",
                               "salesperson_code", "customer_posting_group", "blocked"], "customer")
    customer = customer.withColumn("no", text_code(F.col("no")))
    validate_key(customer, ["no"], "customer")
    customers = customer.select(
        F.lit(COMPANY_KEY).alias("company_key"), relation_key(F.col("no")).alias("customer_key"),
        F.col("no").alias("customer_no"), F.col("name").alias("customer_name"),
        "city", "county", "post_code", "country_region_code", "customer_posting_group", "blocked",
        F.col("salesperson_code").alias("current_salesperson_code"), F.lit(False).alias("is_placeholder"))
    refs = fact.select(F.col("bill_to_customer_key").alias("customer_key")).unionByName(
        fact.select(F.col("sell_to_customer_key").alias("customer_key")))
    customers = add_missing_members(customers, "customer_key", "customer_name", refs)

    item, category = sources["item"], sources["item_category"]
    require_columns(item, ["no", "description", "description_2", "base_unit_of_measure",
                          "item_category_code", "net_weight", "gross_weight", "blocked"], "item")
    require_columns(category, ["code", "description", "parent_category"], "item_category")
    item = item.withColumn("no", text_code(F.col("no")))
    category = category.withColumn("code", text_code(F.col("code")))
    validate_key(item, ["no"], "item")
    validate_key(category, ["code"], "item_category")
    items = item.alias("i").join(category.alias("c"), text_code(F.col("i.item_category_code")) == F.col("c.code"), "left").select(
        F.lit(COMPANY_KEY).alias("company_key"), relation_key(F.col("i.no")).alias("item_key"),
        F.col("i.no").alias("item_no"), F.col("i.description").alias("item_description"),
        F.col("i.description_2").alias("item_description_2"), F.col("i.base_unit_of_measure"),
        F.col("i.item_category_code"), F.col("c.description").alias("item_category_description"),
        F.col("c.parent_category"), F.col("i.net_weight"), F.col("i.gross_weight"), F.col("i.blocked"),
        (text_code(F.col("i.item_category_code")).isNotNull() & (text_code(F.col("i.item_category_code")) != "")
         & F.col("c.code").isNull()).alias("category_missing"), F.lit(False).alias("is_placeholder"))
    items = add_missing_members(items, "item_key", "item_description", fact.select("item_key"))

    rep = sources["salesperson_purchaser"]
    require_columns(rep, ["code", "name", "blocked"], "salesperson_purchaser")
    rep = rep.withColumn("code", text_code(F.col("code")))
    validate_key(rep, ["code"], "salesperson_purchaser")
    reps = rep.select(F.lit(COMPANY_KEY).alias("company_key"), relation_key(F.col("code")).alias("salesperson_key"),
                      F.col("code").alias("salesperson_code"), F.col("name").alias("salesperson_name"),
                      "blocked", F.lit(False).alias("is_placeholder"))
    reps = add_missing_members(reps, "salesperson_key", "salesperson_name", fact.select("salesperson_key"))
    return {"dim_customer": customers, "dim_item": items, "dim_salesperson": reps}


def build_dates(fact, today=None):
    today = today or datetime.now(timezone.utc).date()
    bounds = fact.agg(F.min("posting_date").alias("lo"), F.max("posting_date").alias("hi")).first()
    if bounds["lo"] is None:
        raise ValueError("No posted sales rows; empty fact publication is blocked")
    first_year = min(bounds["lo"].year, today.year)
    last_year = max(bounds["hi"].year, today.year)
    df = spark.range(1).select(F.explode(F.sequence(F.lit(date(first_year, 1, 1)),
                                                   F.lit(date(last_year, 12, 31)))).alias("date"))
    return df.select("date", F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
                     F.year("date").alias("year"), F.quarter("date").alias("quarter"),
                     F.month("date").alias("month_number"), F.date_format("date", "MMMM").alias("month_name"),
                     F.date_format("date", "yyyy-MM").alias("year_month"), F.last_day("date").alias("month_end"),
                     F.dayofmonth("date").alias("day_of_month"))


def check_relationships(outputs):
    fact = outputs["fact_posted_sales"]
    links = [("bill_to_customer_key", "dim_customer", "customer_key"),
             ("sell_to_customer_key", "dim_customer", "customer_key"),
             ("item_key", "dim_item", "item_key"),
             ("salesperson_key", "dim_salesperson", "salesperson_key"),
             ("posting_date_key", "dim_date", "date_key")]
    for fk, table, pk in links:
        target = outputs[table].select(F.col(pk).alias(fk))
        if fact.select(fk).distinct().join(target, fk, "left_anti").limit(1).count():
            raise ValueError(f"Unresolved relationship: {fk} -> {table}.{pk}")


def reconcile_document(lines, headers, actual, document_type):
    # Independent source rollup at document grain; currencies never combined.
    original = lines.alias("l").join(headers.alias("h"), F.col("l.document_no") == F.col("h.no"), "inner")
    groups = ["document_no", "currency_code"]
    raw = original.select(F.col("l.document_no"),
        F.coalesce(text_code(F.col("h.currency_code")), F.lit("")).alias("currency_code"),
        F.col("l.amount").alias("excl"), F.col("l.amount_including_vat").alias("incl"))
    expected = raw.groupBy(*groups).agg(F.count("*").alias("rows"),
                                       F.sum("excl").alias("excl"), F.sum("incl").alias("incl"))
    if document_type == "CREDIT_MEMO":
        expected = expected.withColumn("excl", -F.col("excl")).withColumn("incl", -F.col("incl"))
    observed = actual.filter(F.col("document_type") == document_type).groupBy(*groups).agg(
        F.count("*").alias("rows"), F.sum("signed_amount_excl_vat").alias("excl"),
        F.sum("signed_amount_incl_vat").alias("incl"))
    if expected.exceptAll(observed).limit(1).count() or observed.exceptAll(expected).limit(1).count():
        raise ValueError(f"{document_type}: document counts or exact decimal totals differ from Silver")


def build_gold(sources):
    invoice = build_document(sources["sales_invoice_line"], sources["sales_invoice_header"], "INVOICE")
    credit = build_document(sources["sales_credit_memo_line"], sources["sales_credit_memo_header"], "CREDIT_MEMO")
    fact = invoice.unionByName(credit).persist(StorageLevel.MEMORY_AND_DISK)
    try:
        fact.count()
        validate_key(fact, OUTPUT_KEYS["fact_posted_sales"], "fact_posted_sales")
        for prefix, kind in [("sales_invoice", "INVOICE"), ("sales_credit_memo", "CREDIT_MEMO")]:
            reconcile_document(sources[prefix + "_line"], sources[prefix + "_header"], fact, kind)
        outputs = build_dimensions(sources, fact)
        outputs["dim_date"] = build_dates(fact)
        outputs["fact_posted_sales"] = fact
        for name, frame in outputs.items():
            validate_key(frame, OUTPUT_KEYS[name], name)
        check_relationships(outputs)
        return outputs
    except Exception:
        fact.unpersist()
        raise


def current_version(table):
    return int(spark.sql(f"DESCRIBE HISTORY {table}").select("version").first()["version"])


def load_sources():
    audit_name = f"{SOURCE_LAKEHOUSE}.{SOURCE_SCHEMA}._load_audit"
    audit = spark.table(audit_name)
    sources, manifest = {}, []
    for name in SOURCE_TABLES:
        table = f"{SOURCE_LAKEHOUSE}.{SOURCE_SCHEMA}.{name}"
        # Pin each Delta version to make retries/recomputation within this run stable.
        detail = spark.sql(f"DESCRIBE DETAIL {table}").first()
        if detail["format"].lower() != "delta":
            raise ValueError(f"{table}: expected a persisted Silver Delta table")
        version = current_version(table)
        df = spark.read.format("delta").option("versionAsOf", version).load(detail["location"])
        require_columns(df, ["_silver_run_id"], table)
        latest = (audit.filter(F.col("target_table") == table)
                  .orderBy(F.col("completed_at").desc()).limit(1).collect())
        if not latest or latest[0]["status"] != "SUCCESS":
            raise ValueError(f"{table}: latest Silver audit is missing or unsuccessful")
        run = latest[0]
        if run["effective_mode"] not in {"full", "bootstrap_full"}:
            raise ValueError(f"{table}: Gold v1 expects audited full Silver snapshots")
        reject_rows(df, F.col("_silver_run_id").isNull() | (F.col("_silver_run_id") != run["run_id"]),
                    f"{table}: rows do not match latest audited full load")
        count = df.count()
        if run["saved_rows"] != count:
            raise ValueError(f"{table}: audit row count differs from pinned source")
        sources[name] = df
        manifest.append({"table": table, "version": version, "rows": count,
                         "silver_run_id": run["run_id"], "silver_batch_id": run["batch_id"]})
    return sources, manifest


def assert_sources_unchanged(manifest):
    for entry in manifest:
        if current_version(entry["table"]) != entry["version"]:
            raise ValueError("Silver changed during Gold preparation; run without overlapping writers")


def schema_signature(frame):
    return {field.name: field.dataType.simpleString() for field in frame.schema.fields}


def run_gold():
    import re
    for value in [SOURCE_LAKEHOUSE, SOURCE_SCHEMA, TARGET_LAKEHOUSE, TARGET_SCHEMA]:
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
            raise ValueError("Use simple lakehouse/schema names")
    if SOURCE_LAKEHOUSE == TARGET_LAKEHOUSE:
        raise ValueError("Use a separate Gold lakehouse")
    configure_spark()
    run_id = str(uuid4())
    started = datetime.now(timezone.utc).isoformat()
    print("Gold run:", run_id, "| WRITE_GOLD:", WRITE_GOLD)
    sources, manifest = load_sources()
    print("Pinned Silver sources:")
    for row in manifest:
        print(row["table"], "version", row["version"], "rows", row["rows"])
    outputs = build_gold(sources)
    raw_fact = outputs["fact_posted_sales"]
    cached = []
    try:
        counts = {}
        built_at = datetime.now(timezone.utc)
        for name, frame in list(outputs.items()):
            out = frame.withColumn("_gold_run_id", F.lit(run_id)).withColumn("_gold_processed_at", F.lit(built_at))
            out = out.persist(StorageLevel.MEMORY_AND_DISK)
            cached.append(out)
            counts[name] = out.count()
            outputs[name] = out
            print(f"VALIDATED: {name} | {counts[name]:,} rows")
        print("Totals in document currency (blank currency_code means the company's local currency):")
        (outputs["fact_posted_sales"].groupBy("document_type", "currency_code").agg(
            F.count("*").alias("rows"), F.sum("signed_amount_excl_vat").alias("signed_excl_vat"),
            F.sum("signed_amount_incl_vat").alias("signed_incl_vat"))
         .orderBy("document_type", "currency_code").show(truncate=False))
        for table in ["dim_customer", "dim_item", "dim_salesperson"]:
            print(table, "placeholder members:", outputs[table].filter("is_placeholder").count())
        print("Items with unresolved categories:", outputs["dim_item"].filter("category_missing").count())
        print("Comment lines using header date:", outputs["fact_posted_sales"].filter(
            "line_posting_date_defaulted").count())
        for customer_field in ["bill_to_customer", "sell_to_customer"]:
            print(f"Comment lines using header {customer_field}:",
                  outputs["fact_posted_sales"].filter(customer_field + "_defaulted").count())
        print("Credit rows with negative source amount:", outputs["fact_posted_sales"].filter(
            (F.col("document_type") == "CREDIT_MEMO") & (F.col("source_amount_excl_vat") < 0)).count())
        assert_sources_unchanged(manifest)
        if not WRITE_GOLD:
            print("VALIDATION COMPLETE: five proposed Gold tables passed checks. No Gold tables were written.")
            return
        target_prefix = f"{TARGET_LAKEHOUSE}.{TARGET_SCHEMA}"
        # Target lakehouse must already exist and be accessible/attached in Fabric.
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_prefix}")
        for name, frame in outputs.items():
            target = f"{target_prefix}.{name}"
            if spark.catalog.tableExists(target) and schema_signature(spark.table(target)) != schema_signature(frame):
                raise ValueError(f"{target}: schema drift; review before replacing any Gold table")
        audit_schema = "run_id string, status string, started_at string, completed_at string, source_manifest string, output_counts string, written_tables string, error string"
        audit_table = target_prefix + "._gold_load_audit"
        if not spark.catalog.tableExists(audit_table):
            spark.createDataFrame([], audit_schema).write.format("delta").mode("errorifexists").saveAsTable(audit_table)
        if schema_signature(spark.table(audit_table)) != schema_signature(spark.createDataFrame([], audit_schema)):
            raise ValueError("Gold audit schema differs; resolve before writing")
        def audit(status, written, error=""):
            row = (run_id, status, started, datetime.now(timezone.utc).isoformat(),
                   json.dumps(manifest), json.dumps(counts), json.dumps(written), error[:4000])
            spark.createDataFrame([row], audit_schema).write.format("delta").mode("append").saveAsTable(audit_table)
        written = []
        audit("RUNNING", written)
        try:
            for name in OUTPUT_KEYS:  # Dimensions first, fact last.
                target = f"{target_prefix}.{name}"
                mode = "overwrite" if spark.catalog.tableExists(target) else "errorifexists"
                outputs[name].write.format("delta").mode(mode).saveAsTable(target)
                written.append(name)
                saved = spark.table(target).select(*outputs[name].columns)
                # Exact multiset comparison checks values, duplicates, row counts and lineage.
                if saved.exceptAll(outputs[name]).limit(1).count() or outputs[name].exceptAll(saved).limit(1).count():
                    raise ValueError(f"{target}: saved contents differ from validated data")
                print("SAVED AND VERIFIED:", target)
            audit("SUCCESS", written)
        except Exception as error:
            try:
                audit("FAILED", written, str(error))
            except Exception:
                print("Could not record failure; the RUNNING audit may remain. Inspect pipeline logs.")
            raise
        print("COMPLETE: all five Gold tables written, verified, and audited.")
    finally:
        for frame in cached:
            frame.unpersist()
        raw_fact.unpersist()


## 3. Run

This is the only execution cell. Run all cells in order after changing configuration.

In [ ]:
run_gold()